In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('anime.csv')

In [3]:
df

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266
...,...,...,...,...,...,...,...
12289,9316,Toushindai My Lover: Minami tai Mecha-Minami,Hentai,OVA,1,4.15,211
12290,5543,Under World,Hentai,OVA,1,4.28,183
12291,5621,Violence Gekiga David no Hoshi,Hentai,OVA,4,4.88,219
12292,6133,Violence Gekiga Shin David no Hoshi: Inma Dens...,Hentai,OVA,1,4.98,175


In [4]:
df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [5]:
# Convert episodes to numeric (dataset sometimes contains '?')
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')

# Fill missing values
df['genre'] = df['genre'].fillna("Unknown")
df['type'] = df['type'].fillna("Unknown")
df['rating'] = df['rating'].fillna(df['rating'].mean())
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

df.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

In [6]:
# Display Basic Stats
print("Total Anime:", df.shape[0])
print("Total Features:", df.shape[1])

print("\nAnime Types:\n", df['type'].value_counts())
print("\nTop Genres:\n", df['genre'].value_counts().head())

Total Anime: 12294
Total Features: 7

Anime Types:
 type
TV         3787
OVA        3311
Movie      2348
Special    1676
ONA         659
Music       488
Unknown      25
Name: count, dtype: int64

Top Genres:
 genre
Hentai                   823
Comedy                   523
Music                    301
Kids                     199
Comedy, Slice of Life    179
Name: count, dtype: int64


In [7]:
# Feature Extraction

In [8]:
# Combine Text Features (genre + type)
df['text_features'] = df['genre'] + " " + df['type']
df[['name', 'text_features']].head()

,name,text_features
0,Kimi no Na wa.,"Drama, Romance, School, Supernatural Movie"
1,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili..."
2,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S..."
3,Steins;Gate,"Sci-Fi, Thriller TV"
4,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S..."


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
#### TF-IDF Vectorization (For Text)
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df['text_features'])

tfidf_matrix.shape

(12294, 52)

In [11]:
# Normalize Numerical Features
scaler = MinMaxScaler()

df[['rating_scaled', 'episodes_scaled']] = scaler.fit_transform(
    df[['rating', 'episodes']]
)

df[['rating_scaled', 'episodes_scaled']].head()


,rating_scaled,episodes_scaled
0,0.924370,0.000000
1,0.911164,0.034673
2,0.909964,0.027518
3,0.900360,0.012658
4,0.899160,0.027518


In [12]:
# Merge All Features into Final Feature Matrix
# Numerical features
numeric_features = df[['rating_scaled', 'episodes_scaled']].values

# Combine TF-IDF + numeric
from scipy.sparse import hstack

final_features = hstack([tfidf_matrix, numeric_features])

final_features.shape

(12294, 54)

In [13]:
# Recommendation Using Cosine Similarity

In [ ]:
# Compute Cosine Similarity Matrix
similarity_matrix = cosine_similarity(final_features, final_features)
similarity_matrix.shape

In [ ]:
# Recommendation Function
def recommend_anime(title, similarity_threshold=0.2, top_n=10):
    if title not in df['name'].values:
        return ["Anime not found in dataset."]

    # Get index of target anime
    index = df[df['name'] == title].index[0]

    # Get similarity scores
    scores = list(enumerate(similarity_matrix[index]))

    # Filter by similarity threshold
    filtered = [(i, score) for i, score in scores if score >= similarity_threshold and i != index]

    # Sort by similarity score
    filtered = sorted(filtered, key=lambda x: x[1], reverse=True)

    # Get recommended anime titles
    recommended_indices = [i for i, s in filtered][:top_n]
    
    return df['name'].iloc[recommended_indices].tolist()


In [ ]:
# Test Recommendation System
recommend_anime("One Piece", similarity_threshold=0.25, top_n=10)


### 1. Difference between User-Based and Item-Based Collaborative Filtering
#### User-Based CF: Recommends items by finding users with similar preferences to the target user and suggesting items those similar users liked.

#### Item-Based CF: Recommends items by finding items similar to the ones the user has already liked and suggesting those similar items.

#### Key Difference: User-based focuses on similar users, while item-based focuses on similar items (item-based is usually more scalable and stable).

### 2. What is Collaborative Filtering and How Does It Work
#### Collaborative Filtering: A recommendation technique that predicts user interests based on past interactions (ratings, clicks, purchases) from multiple users.
#### How it Works:
#### Collect user–item interaction data.
#### Identify patterns or similarities (between users or items).
#### Generate recommendations based on these similarities without needing item content information.